Notebook to validate if platform models sent predictions for all geocodes and validation sets per challenge.

In [1]:
import numpy as np
import pandas as pd
import mosqlient as mosq
from epiweeks import Week

In [2]:
import os
from dotenv import load_dotenv

# Access the environment variables
api_key = os.getenv('api_key')

In [3]:
def validate_preds(df_preds_all, disease='A90', col='adm_1'): 
    # 1. Filtrar pela doença selecionada (.copy() evita avisos de SettingWithCopyWarning)
    df_preds_info = df_preds_all.loc[df_preds_all.disease == disease].copy()
    df_preds_info = df_preds_info.loc[df_preds_info.adm_1 != 32]
    df_preds_info = df_preds_info.loc[df_preds_info.start < Week(2026, 41).startdate()]

    # 2. CORREÇÃO: Aplicar a exceção do adm_1 == 32 se estivermos analisando estados
    if col == 'adm_1':
        df_preds_info = df_preds_info.loc[(df_preds_info.adm_2.isna())]

    # 3. CORREÇÃO: Mapeamento de regras seguro (evita UnboundLocalError e remove o '&')
    valid_configs = {
        ('adm_1', 'A90'): {'size': 26, 'challenge': 'dengue_state'},
        ('adm_1', 'A92.0'): {'size': 26, 'challenge': 'chik_state'},
        ('adm_2', 'A90'): {'size': 15, 'challenge': 'dengue_city'},
        ('adm_2', 'A92.0'): {'size': 10, 'challenge': 'chik_city'},
    }

    # Valida se os parâmetros passados existem nas regras de negócio
    if (col, disease) not in valid_configs:
        raise ValueError(f"A combinação de coluna '{col}' e doença '{disease}' não é válida ou configurada.")

    # Resgata os valores corretos com segurança
    config = valid_configs[(col, disease)]
    size = config['size']
    challenge = config['challenge']

    # 4. Agrupamento detalhado por modelo, doença e localidade
    grupo_detalhado = (
        df_preds_info.groupby(['model_name', 'disease', col])
        .size()
        .reset_index(name='qtd_validacoes')
    )

    # 5. Agrupamento final para gerar o relatório
    relatorio_validacao = (
        grupo_detalhado.groupby(['model_name', 'disease'])
        .agg(
            qtd_adm=(col, 'nunique'),
            todos_com_4_validaçoes=('qtd_validacoes', lambda x: (x == 4).all())
        )
        .reset_index()
    )

    # Trata o caso de o retorno ser um DataFrame vazio para não quebrar a lógica
    if relatorio_validacao.empty:
        return pd.DataFrame(columns=['model_name', 'disease', 'qtd_adm', 'todos_com_4_validaçoes', 'challenge', 'dados_corretos'])

    # 6. Adiciona as colunas de validação final
    relatorio_validacao['challenge'] = challenge
    relatorio_validacao['dados_corretos'] = (
        (relatorio_validacao['qtd_adm'] == size) & 
        (relatorio_validacao['todos_com_4_validaçoes'])
    )

    return relatorio_validacao

def load_preds(api_key = api_key, model_name = 'str'): 

    preds = mosq.get_predictions(api_key = api_key, model_name= model_name)

    if len(preds) >0 :
        df_info = pd.DataFrame([
            {   "pred": pred, 
                "model_name": pred.model.repository,
                "id": pred.id, 
                "commit": pred.commit, 
                "disease": pred.disease,
                "adm_1": pred.adm_1,
                "adm_2": pred.adm_2, 
                "start": pred.start,
                "end": pred.end,
                "wis": pred.scores.get("wis")
                if pred.scores is not None else None,
                "published": pred.published }
            for pred in preds
        ])

        df_info = df_info.loc[df_info.published == True]

        df_info['validation'] = None 

        df_info.loc[(df_info.start == Week(2022, 41).startdate()) & (df_info.end == Week(2023, 40).startdate()), 'validation'] = 1 
        df_info.loc[(df_info.start == Week(2023, 41).startdate()) & (df_info.end == Week(2024, 40).startdate()), 'validation'] = 2 
        df_info.loc[(df_info.start == Week(2024, 41).startdate()) & (df_info.end == Week(2025, 40).startdate()), 'validation'] = 3 
        df_info.loc[(df_info.start == Week(2025, 41).startdate()) & (df_info.end == Week(2026, 40).startdate()), 'validation'] = 4 

        return df_info 


    else: 
        print(f'Nenhum previsão registrada para o modelo {model_name}')





Load 3rd IMDC models: 

In [4]:
models = mosq.get_models(api_key = api_key, imdc_year = 2026)

models

[asgouveiaa/3rd_imdc_fiocruz_mard,
 BentoLab-DiseaseDynamics/3rd_imdc_cornell_bentolab,
 lsbastos/3rd_imdc_procc_bb_model,
 dievillano/3rd_imdc_bsc_ghr,
 marciomacielbastos/3rd_imdc_fgv_sakhal,
 americocunhajr/3rd_imdc_lncc_clidengo26chikungunya,
 americocunhajr/3rd_imdc_lncc_clidengo26dengue,
 Ricafya/3rd_imdc_afya_ric,
 SungmokJung/3rd_imdc_nus_nus-cerm,
 eduardocorrearaujo/3rd_imdc_emap_lstm_muni,
 graeme-dor/3rd_imdc_ceri_returnoftheforecast,
 kamrul28890/3rd_imdc_purdue_neuralearth,
 eduardocorrearaujo/3rd_imdc_emap_lstm,
 InfraMIND-models/3rd_imdc_ifgw_inframind-proteus,
 EzequielEBS/3rd_imdc_emap_epidematicos_prophet_fixed,
 EzequielEBS/3rd_imdc_emap_epidematicos_sarimax_fixed,
 Dududidicao99/3rd_imdc_pucrio_arbocaster,
 pesquefGH/3rd_imdc_lncc_ARp26_chikungunya,
 pesquefGH/3rd_imdc_lncc_surge_model26_dengue,
 pesquefGH/3rd_imdc_lncc_lncc_arp26_dengue,
 mattiamazzoli/3rd_imdc_isi_isi-dengue,
 blaiate/3rd_imdc_-unifesp-_-4mosqueteiras-,
 eduardocorrearaujo/3rd_imdc_emap_example,


In [5]:
list_df_val = []

for model in models: 
    
    df_preds_info = load_preds(api_key=api_key, model_name=model.repository.split('/')[1])

    if df_preds_info is not None: 

        list_df_val.append(validate_preds(df_preds_info, disease = 'A90', col = 'adm_1'))
            
        list_df_val.append(validate_preds(df_preds_info, disease = 'A92.0', col = 'adm_1'))

        list_df_val.append(validate_preds(df_preds_info, disease = 'A90', col = 'adm_2'))

        list_df_val.append(validate_preds(df_preds_info, disease = 'A92.0', col = 'adm_2'))
    
df_val = pd.concat(list_df_val, ignore_index= True)
df_val.head()

100%|██████████| 1/1 [00:01<00:00,  1.52s/requests]


Nenhum previsão registrada para o modelo 3rd_imdc_unesp_recogna_TTM


100%|██████████| 2/2 [00:03<00:00,  1.59s/requests]


Nenhum previsão registrada para o modelo 3rd_imdc_rki_rki_zki_ph_chronos


100%|██████████| 4/4 [00:28<00:00,  7.18s/requests]


Nenhum previsão registrada para o modelo ADCaptura-container


,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos
0,asgouveiaa/3rd_imdc_fiocruz_mard,A90,26,True,dengue_state,True
1,BentoLab-DiseaseDynamics/3rd_imdc_cornell_bent...,A90,26,True,dengue_state,True
2,lsbastos/3rd_imdc_procc_bb_model,A90,26,True,dengue_state,True
3,lsbastos/3rd_imdc_procc_bb_model,A92.0,26,True,chik_state,True
4,dievillano/3rd_imdc_bsc_ghr,A90,26,True,dengue_state,True


In [6]:
df_val.to_csv('predictions/validate_models.csv', index = False)